# Атака на широковещательный вариант RSA (RSA Broadcast attack)
Представьте, что Вы используете открытую экспоненту $e=3$ (не так давно она была достаточно популярна), но вы зашифровываете только сообщения, которые переполняют модуль $N$ при возведении в третью степень, так что нельзя просто взять и извлечь корень третьей степени. Правда ли это безопасно? Есть довольно простой сценарий, который наглядно показывает проблему небольшой открытой экспоненты.
Представьте, что пользователь использует RSA, чтобы посылать сообщения на несколько серверов (все с экспонентой 3, но разными модулями $N$). Пусть таких серверов 3. Одно и то же сообщение шифруется 3 раза: с ключами первого, второго и третьего серверов.
$$C_1=M^{3}\ \mathit{mod}\ N_1$$
$$C_2=M^{3}\ \mathit{mod}\ N_2$$
$$C_3=M^{3}\ \mathit{mod}\ N_3$$
Почти невозможно дешифровать каждый из шифротекстов $C_{i}$ по одиночке, но с тремя Марвин(или Мэллори, как вам больше нравится) может решить эту задачу.

## Китайская теорема об остатках
Если известны остатки от Евклидова деления целого числа $n$ на несколько целых чисел, то можно определить уникальный остаток от деления $n$ на произведение этих целых чисел, при условии, что все делители (модули, по которым брали остатки) попарно простые. 

Как мы можем использовать эту теорему? Во-первых, нам необходимо проверить, что все делители  ($N_1, N_2, N_3$) взаимно (попарно) простые. Но, если вдруг они не взаимно простые, то наибольший общий делитель двух из них больше единицы и либо они равны (и мы не можем их использовать), либо они имеют в составе одно и то же простое число - их $НОД$. В последнем случае мы и так можем расшифровать сообщение. Так что будем считать, что они попарно взаимнопростые. Это значит, что существует такой $X$, что:
$$ X\lt N_1 N_2 N_3$$
$$ X = C_1\ \mathit{mod}\ N_1 $$
$$ X = C_2\ \mathit{mod}\ N_2 $$
$$ X = C_3\ \mathit{mod}\ N_3 $$
и такой $X$ уникален.

Давайте рассмотрим $C=M^{3}$. Так как $M\lt N_1$ и $M\lt N_2$ и $M\lt N_3$, то $C=M^3\lt N_1N_2N_3$. А ещё:
$$ C = C_1\ \mathit{mod}\ N_1 $$
$$ C = C_2\ \mathit{mod}\ N_2 $$
$$ C = C_3\ \mathit{mod}\ N_3 $$

Так что, используя китайскую теорему об остатках, можно найти $C$ и всё, что останется сделать - это извлечь кубический корень.

## Как получить C?

Пусть $N_i, i=1,k$ - модули, а $c_i, i=1,k$ - остатки по делению. $N=N_1N_2...N_k$, а $M_i=N/N_i$
Тогда $C=(\sum_{i=1}^{k}C_iM_i(M_i^{-1}\ \mathit{mod}\ N_i))\ \mathit{mod}\ N$

Воспользуйтесь выражением и вытащите флаг из трёх зашифрованных сообщений, полученных с сервера. Удачи!

In [12]:
import socket
import re
from Crypto.Util.number import inverse, long_to_bytes, bytes_to_long

class VulnServerClient:
    def __init__(self,show=True):
        """Initialization, connecting to server"""
        self.s=socket.socket(socket.AF_INET,socket.SOCK_STREAM)
        self.s.connect(('cryptotraining.zone',1339))
        if show:
            print (self.recv_until().decode())
    def recv_until(self,symb=b'\n>'):
        """Receive messages from server, by default till new prompt"""
        data=b''
        while True:
            
            data+=self.s.recv(1)
            if data[-len(symb):]==symb:
                break
        return data
    def get_public_keys(self,show=True):
        """Receive public keys from the server"""
        self.s.sendall('public\n'.encode())
        response=self.recv_until().decode()
        if show:
            print (response)
        e1=int(re.search('(?<=e1: )\d+',response).group(0))
        N1=int(re.search('(?<=N1: )\d+',response).group(0))
        e2=int(re.search('(?<=e2: )\d+',response).group(0))
        N2=int(re.search('(?<=N2: )\d+',response).group(0))
        e3=int(re.search('(?<=e3: )\d+',response).group(0))
        N3=int(re.search('(?<=N3: )\d+',response).group(0))
       
        return [(e1,N1),(e2,N2),(e3,N3)]
    
    def get_ciphertexts(self,show=True):
        """Receive ciphertexts from the server"""
        self.s.sendall('ciphertext\n'.encode())
        response=self.recv_until().decode()
        if show:
            print (response)
        c1=bytes_to_long(bytes.fromhex(re.search('(?<=ciphertext1: )[0-9a-f]+',response).group(0)))
        c2=bytes_to_long(bytes.fromhex(re.search('(?<=ciphertext2: )[0-9a-f]+',response).group(0)))
        c3=bytes_to_long(bytes.fromhex(re.search('(?<=ciphertext3: )[0-9a-f]+',response).group(0)))
        return (c1,c2,c3)
    
    def __del__(self):
        self.s.close()

<>:27: SyntaxWarning: invalid escape sequence '\d'
<>:28: SyntaxWarning: invalid escape sequence '\d'
<>:29: SyntaxWarning: invalid escape sequence '\d'
<>:30: SyntaxWarning: invalid escape sequence '\d'
<>:31: SyntaxWarning: invalid escape sequence '\d'
<>:32: SyntaxWarning: invalid escape sequence '\d'
<>:27: SyntaxWarning: invalid escape sequence '\d'
<>:28: SyntaxWarning: invalid escape sequence '\d'
<>:29: SyntaxWarning: invalid escape sequence '\d'
<>:30: SyntaxWarning: invalid escape sequence '\d'
<>:31: SyntaxWarning: invalid escape sequence '\d'
<>:32: SyntaxWarning: invalid escape sequence '\d'
/var/folders/jm/fq05587x6d787nhsnkqr2zy40000gp/T/ipykernel_54202/1008483078.py:27: SyntaxWarning: invalid escape sequence '\d'
  e1=int(re.search('(?<=e1: )\d+',response).group(0))
/var/folders/jm/fq05587x6d787nhsnkqr2zy40000gp/T/ipykernel_54202/1008483078.py:28: SyntaxWarning: invalid escape sequence '\d'
  N1=int(re.search('(?<=N1: )\d+',response).group(0))
/var/folders/jm/fq05587x6d

In [2]:
vs=VulnServerClient()
pk_list=vs.get_public_keys()
(c1,c2,c3)=vs.get_ciphertexts()

Welcome to RSA broadcast task
Available commands:
help - print this help
public - show public keys
ciphertext - show ciphertexts 
quit - quit
>
e1: 3
N1: 20287982006618431876793244706487063574769448388426702838915722457901061849764724362603953179532539827554365329323483013276648891796378507103593000936891322846098718727133325953848182431476448546554772557085987908949429403596359635314342612889908898272272322173341141651567301687427226491229442385493785765699715986462483124423171652756203919879715705590771525305446788322512844427648822922682205388423707896633544989180321378196798302096862401020103125458117084856441433418990681274327810046291890889882496621395403208554291777227706389355365678885157011052465893070731243360783878242460400701935579682716345964783581
e2: 3
N2: 19461826656993775602233892694656909792660373232000384211810816270789889052589469816096210883392037679550505993511270364500797371312576866213808413522877454484246665882296260574836114845774508160454919015698629693496502

Решение

1. Получаем данные: шифр-тексты и $N_{i}$

In [11]:
N1, N2, N3 = (N_i[1] for N_i in pk_list)
print(f"Data types: {type(N1)}, {type(N2)}, {type(N3)}")
print(f"Data size: {N1.bit_length()}, {N2.bit_length()}, {N3.bit_length()}")

Data types: <class 'int'>, <class 'int'>, <class 'int'>
Data size: 2048, 2048, 2048


2. Далее, проверим на попарную взаимную простоту

In [21]:
import math
from itertools import combinations
from typing import List 

def pair_gcd(numbers: List, r = 2):
    pairs = list(combinations(numbers, r))
    for pair in pairs:
        if math.gcd(*pair) != 1:
            print(pair)
            return False
    
    return True

print(f"Pairwise mutuall simple?: {pair_gcd([N1, N2, N3])}")

Pairwise mutuall simple?: True


Условие выполнено. Напишем функцию получения ```C:```

In [46]:
def get_CTO_solution(cipher_texts: List, modules: List) -> int:
    
    C = 0

    N = math.prod(modules)
    
    for N_i, C_i in zip(modules, cipher_texts):

        M_i = N // N_i
        C_ = C_i * M_i * inverse(M_i, N_i) % N
        C += C_
        C %= N
    
    return C

M_3 = get_CTO_solution(cipher_texts=[c1, c2, c3], modules=[N1, N2, N3])

Извлекаем корень

In [47]:
from sympy import integer_nthroot

M, _ = integer_nthroot(M_3, 3)
flag = long_to_bytes(M)
print(flag)

b"Congratulations! Here is your flag: CRYPTOTRAINING{3_p30pl3_c4n7_k33p_4_s3cr3t}. Also some placeholder text, because I need it to be around 100 bytes to overflow the modulus, since I don't want to use paddings. YET"
